In [1]:
import pandas as pd 
import numpy as np 


In [3]:
df=pd.read_csv('labeled_data.csv')
df.isna().sum()

Unnamed: 0            0
count                 0
hate_speech           0
offensive_language    0
neither               0
class                 0
tweet                 0
dtype: int64

In [4]:
df.head()

,Unnamed: 0,count,hate_speech,offensive_language,neither,class,tweet
0,0,3,0,0,3,2,!!! RT @mayasolovely: As a woman you shouldn't...
1,1,3,0,3,0,1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...
2,2,3,0,3,0,1,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...
3,3,3,0,2,1,1,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...
4,4,6,0,6,0,1,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...


In [20]:
data=df[['tweet','class']]
data.head()

,tweet,class
0,!!! RT @mayasolovely: As a woman you shouldn't...,2
1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...,1
2,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...,1
3,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...,1
4,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...,1


In [21]:
import re

def clean(text):
    text=text.lower()
    text=re.sub(r'http\S+|www\S+','',text)
    text=re.sub(r'@\w+', '', text)
    text=re.sub(r'#\w+', '', text)
    text=re.sub(r'[^a-z\s]', '', text)
    text=re.sub(r'\s+', ' ', text).strip()     
    return text

data['cleaned_tweet']=data['tweet'].apply(clean)

C:\Users\Dell\AppData\Local\Temp\ipykernel_6228\643930702.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['cleaned_tweet']=data['tweet'].apply(clean)


In [22]:
data.head()

,tweet,class,cleaned_tweet
0,!!! RT @mayasolovely: As a woman you shouldn't...,2,rt as a woman you shouldnt complain about clea...
1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...,1,rt boy dats coldtyga dwn bad for cuffin dat ho...
2,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...,1,rt dawg rt you ever fuck a bitch and she start...
3,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...,1,rt she look like a tranny
4,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...,1,rt the shit you hear about me might be true or...


In [23]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout



In [24]:
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(df['tweet'])

sequences = tokenizer.texts_to_sequences(df['tweet'])
padded_sequences = pad_sequences(sequences, maxlen=100)

In [25]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split( padded_sequences, df['class'], test_size=0.2, random_state=42)

In [26]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

num_classes = 3  # normal, offensive, hate

model = Sequential()
model.add(Embedding(input_dim=10000, output_dim=64, input_shape=(100,)))  # instead of input_length
model.add(LSTM(64))
model.add(Dropout(0.5))
model.add(Dense(3, activation='softmax'))  # 3 for multiclass

model.summary()  # Now shows full model


c:\Users\Dell\OneDrive\Desktop\python programs\tf-env\lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 100, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 673,219 (2.57 MB)

 Trainable params: 673,219 (2.57 MB)

 Non-trainable params: 0 (0.00 B)

In [27]:
model.compile(
    loss='sparse_categorical_crossentropy',  # or 'categorical_crossentropy' if one-hot
    optimizer='adam',
    metrics=['accuracy']
)


In [28]:
print(np.unique(y_train))
# Output: [0 1 2]


[0 1 2]


In [29]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
x_train = np.array(x_train)
y_train = np.array(y_train)

model.fit(
    x_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop]
)


Epoch 1/10
558/558 ━━━━━━━━━━━━━━━━━━━━ 24s 39ms/step - accuracy: 0.8127 - loss: 0.5639 - val_accuracy: 0.9052 - val_loss: 0.2817
Epoch 2/10
558/558 ━━━━━━━━━━━━━━━━━━━━ 26s 46ms/step - accuracy: 0.9234 - loss: 0.2340 - val_accuracy: 0.8996 - val_loss: 0.2820
Epoch 3/10
558/558 ━━━━━━━━━━━━━━━━━━━━ 26s 47ms/step - accuracy: 0.9456 - loss: 0.1670 - val_accuracy: 0.8996 - val_loss: 0.3169


In [40]:
inverse_label_mapping = {
    0: 'Hate speech',
    1: 'Offensive language',
    2: 'No hate and offensive'
}



# ⚠️ Assume tokenizer and label_encoder are already fitted
#      and 'model' is trained and ready
def predict_tweet(tweet):
    seq = tokenizer.texts_to_sequences([tweet])
    padded = pad_sequences(seq, maxlen=100)

    pred = model.predict(padded)
    class_idx = np.argmax(pred, axis=1)[0]
    class_label = inverse_label_mapping[class_idx]

    print(f"\nTweet: {tweet}")
    print(f"Predicted Class: {class_label}")


# 🔁 Ask user for input (you can wrap this in a loop if needed)
user_tweet = input("Enter a tweet to classify: ")
predict_tweet(user_tweet)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step

Tweet: all jews shoul be killed
Predicted Class: No hate and offensive


In [41]:
df['class'].value_counts()


class
1    19190
2     4163
0     1430
Name: count, dtype: int64